<a href="https://colab.research.google.com/github/michelle-lo/Sitcom-Topic-Modeling/blob/main/Brooklyn99_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-reqs

In [ ]:
!pip install gensim
!pip install bertopic
!pip install bertopic[sentence-transformers]
!pip install pyLDAvis

In [ ]:
import re
import pandas as pd, json
import json, os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

import spacy
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel, Phrases, TfidfModel
from gensim.models.phrases import Phraser

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import math

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Data

Source: https://github.com/atharva-naik/brooklyn99-dataset/tree/main/plain-text/season-1

Brief description of dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
data_path = "/content/drive/MyDrive/CSDS 601/dataset"

In [ ]:
rows = []

for season in sorted(os.listdir(data_path)):

  season_path = os.path.join(data_path, season)
  if not os.path.isdir(season_path):
      continue

  for episode_file in sorted(os.listdir(season_path)):
      episode_path = os.path.join(season_path, episode_file)
      with open(episode_path, "r", encoding="utf-8") as f:
          episode = json.load(f)

      for line in episode:
        rows.append({
            "season": season,
            "episode": episode_file,
            "text": line
        })

df = pd.DataFrame(rows)
print(df.shape)
df

## Helper methods

In [ ]:
def create_chunked_corpus(df, chunk_size, text_col='text'):
    chunks = []
    for (season, episode), group in df.groupby(["season", "episode"]):
        lines = group[text_col].tolist()
        for i in range(0, len(lines), chunk_size):
            chunk = " ".join(lines[i:i+chunk_size])
            if len(chunk.split()) > 5:
                chunks.append({
                    "season": season,
                    "episode": episode,
                    "text": chunk
                })
    return pd.DataFrame(chunks)

In [ ]:
# building a comprehensive stop_words list

english_stopwords = set(stopwords.words("english"))
b99_stopwords = {
    # character names
    "jake", "peralta", "amy", "santiago", "rosa", "diaz",
    "charles", "boyle", "gina", "linetti", "raymond", "holt",
    "terry", "jeffords", "scully", "hitchcock",
    # dialogue words
    "yeah", "okay", "hey", "uh", "um", "gonna", "wanna",
    "got", "know", "like", "right", "oh", "well", "just",
    "said", "say", "go", "get", "one", "think", "yes",
    "no", "ok", "sure", "wait", "look", "come",
    "want", "need", "mean", "tell", "good", "really"
    # after testing
    "guy", "time", "thing", "man", "way", "little",
    "bad", "day", "people", "fine", "year", "let",
    "cool", "big", "wrong", "real", "new", "nice",
    "lot", "fun", "old", "ill", "thank", "great",
    "sorry", "charle", "first", "possible", "minute",
    "hour", "hell", "weird",
}

stop_words = english_stopwords.union(b99_stopwords)

## Data Cleaning

In [ ]:
# Tuple of (pattern, replacement) pairs
clean_patterns = [
    (r"\[.*?\]", ""),    # remove stage directions [giggles]
    (r"\(.*?\)", ""),    # remove stage directions (laughs)
    (r"\b[A-Z]{2,}\b:", ""), # remove character directions e.g. PERALTA:
    (r"♪", ""),          # remove music symbols
    # Subtitle metadata
    # Matches sync/fixed/corrected metadata with separator (&amp; or and)
    (r"(?i)(sync(ed)?|fixed)\s*(&amp;|and)\s*(sync(ed)?|corrections?|corrected).*", ""),
    # Matches sync/fixed metadata without separator (e.g. "Sync corrections by", "Synchronized by")
    (r"(?i)(sync(ed|ronized)?|fixed)\s*(corrections?\s*by|by).*", ""),
    (r"(?i)\s*ripped\s*by\s*\S+.*", ""),
    # Remove the closing credits line
    (r"(?i)-?\s*not a doctor[.,]?\s*-?\s*shh[.!]?.*", ""),
    (r"^>+\s*", ""),    # remove leading > at start of line
]

def clean_text(text):
    for pattern, replacement in clean_patterns:
        text = re.sub(pattern, replacement, text)
    text = text.strip()  # remove leading/trailing whitespace
    return text

# Clean the data
df["clean_text"] = df["text"].apply(clean_text)

# Filter empty lines
df = df[df["clean_text"].str.replace(r"[^\w]", "", regex=True).str.len() > 0]

df

In [ ]:
def find_lines_containing(df, search_string, column="text"):
    mask = df[column].str.contains(search_string, case=False, na=False)
    results = df[mask]
    print(f"Found {len(results)} lines containing '{search_string}'")
    return results

find_lines_containing(df, "ripped by")

In [ ]:
df = df[['season', 'episode', 'clean_text']].copy().rename(columns={'clean_text': 'text'})
df

# LDA

For LDA, [insert reason why longer documents are better] we use episodes since it works better on longer documents with more diverse set of vocabulary. blah *blah*

1. Lowercase everything
2. Remove whitespace, punctuation, numbers
3. Tokenize
4. Remove stop words (english stop words, show sepcific words like character names and fillter dialogue words)
5. Lemmatize (keep to roots) vs stemming
6. Filter by token length and frequency, and then filter extremes
7. Build Bag of Words

## Baseline LDA

In [ ]:
lda_df = (
    df.groupby(["season", "episode"])["text"]
    .apply(lambda lines: " ".join(lines))
    .reset_index()
)

print(f"Total episodes:", len(lda_df))
lda_df.head()

In [ ]:
def preprocess_for_lda(text):
  # make lowercase and remove punctuation
  text = text.lower()
  text = re.sub(r"[^a-z\s]", "", text)

  # lemmatize and filter via spaCy
  docs = nlp(text)
  tokens = [
      token.lemma_ for token in docs
      if token.is_alpha # letters only
      and token.lemma_ not in stop_words # and not a stop word
      and len(token.lemma_) > 2 # longer than 2 characters (like to, an, ok)
      and token.lemma_ != "-PRON-" # like he, she, they
  ]
  return tokens

# Preprocess episodes for LDA
lda_docs = [preprocess_for_lda(text) for text in lda_df["text"]]

lda_docs[1][:20]

In [ ]:
dictionary = corpora.Dictionary(lda_docs)

dictionary.filter_extremes(
  no_below=5, # words must appear in at least 2 episodes
  no_above=0.8, # words can't appear in more than 80% of episodes
  keep_n=10000
)

# convert into bag of words
corpus = [dictionary.doc2bow(doc) for doc in lda_docs]

print(f"Dictionary size: {len(dictionary)}")
print(f"Corpus size: {len(corpus)}")

lda_docs[1][:20]

In [ ]:
# training the LDA model
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    passes=10,
    random_state=42
)

for idx, topic in lda_model.print_topics(num_words=10):
  print(f"Topic {idx}: {topic}")

In [ ]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=lda_docs,
    dictionary=dictionary,
    coherence="c_v"
)

print(f"Coherence Score: {coherence_model.get_coherence()}")

In [ ]:
# tuning number of topics and picking the one with the highest coherence

# specify range of topics
min_topics = 5
max_topics = 50
step_size = 5
topics_range = range(min_topics, max_topics + 1, step_size)

# grid search over number of topics
coherence_scores = []

for num_topics in topics_range:
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        passes=15,
        random_state=42
    )
    cm = CoherenceModel(
        model=lda_model,
        texts=lda_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"Topics: {num_topics}, Coherence: {score:.4f}")

# find optimal
optimal_num_topics = list(topics_range)[np.argmax(coherence_scores)]
print(f"\nOptimal Number of Topics: {optimal_num_topics}")
print(f"Coherence Score: {max(coherence_scores):.4f}")

# plot
plt.figure(figsize=(10, 6))
plt.plot(topics_range, coherence_scores, marker="o", color="b", label="Coherence Score")
plt.axvline(x=optimal_num_topics, color="r", linestyle="--", label=f"Optimal: {optimal_num_topics}")
plt.xlabel("Number of Topics")
plt.ylabel("Coherence Score")
plt.title("Number of Topics vs Coherence Score")
plt.xticks(topics_range)
plt.legend()
plt.grid(True)
plt.show()

## Enhanced LDA

In [ ]:
# chunk by scene since each episode can contain different plot lines
def create_chunked_corpus(df, chunk_size, text_col='text'):
    chunks = []
    for (season, episode), group in df.groupby(["season", "episode"]):
        lines = group[text_col].tolist()
        for i in range(0, len(lines), chunk_size):
            chunk = " ".join(lines[i:i+chunk_size])
            if len(chunk.split()) > 5:
                chunks.append({
                    "season": season,
                    "episode": episode,
                    "text": chunk
                })
    return pd.DataFrame(chunks)

# Create the new enhanced_lda_df
enhanced_lda_df = create_chunked_corpus(df, chunk_size=50)

print(f"Number of chunks: {len(enhanced_lda_df)}")
enhanced_lda_df.head()

In [ ]:
enhanced_stop_words = stop_words.copy()

def preprocess_for_enhanced_lda(text, stop_words):
    # make lowercase and remove punctuation
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)

    # lemmatize and filter via spaCy — nouns and adjectives only
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if token.is_alpha
        and token.lemma_ not in stop_words
        and len(token.lemma_) > 2
        and token.lemma_ != "-PRON-"
        and token.pos_ in ("NOUN", "PROPN", "ADJ")  # restrict to nouns + adjectives
    ]
    return tokens

def use_enhanced_lda(df, stop_words=enhanced_stop_words):

  # Preprocess episodes for enhanced LDA
  enhanced_lda_docs = [preprocess_for_enhanced_lda(text, stop_words) for text in enhanced_lda_df["text"]]

  # Build bigrams
  bigram = Phrases(enhanced_lda_docs, min_count=3, threshold=5)
  bigram_mod = Phraser(bigram)

  # Build trigrams on top of bigrams
  trigram = Phrases(bigram[enhanced_lda_docs], min_count=3, threshold=5)
  trigram_mod = Phraser(trigram)

  # Apply bigrams and trigrams
  enhanced_lda_docs = [trigram_mod[bigram_mod[doc]] for doc in enhanced_lda_docs]

  return enhanced_lda_docs

# Preview
enhanced_lda_docs = use_enhanced_lda(enhanced_lda_df)
print(enhanced_lda_docs[1][:20])



In [ ]:
# generic verbs and fillers
enhanced_stop_words.update({
    "guy", "time", "thing", "man", "way", "little",
    "bad", "day", "people", "fine", "year", "let",
    "cool", "big", "wrong", "real", "new", "nice",
    "lot", "fun", "old", "ill", "thank", "great",
    "sorry", "charle", "first", "possible", "minute",
    "hour", "hell", "weird",
})

# use the updated stop words
enhanced_lda_docs = use_enhanced_lda(enhanced_lda_df, stop_words=enhanced_stop_words)
enhanced_lda_docs[1][:20]


In [ ]:
# using tfidif


# build dictionary
enhanced_dictionary = corpora.Dictionary(enhanced_lda_docs)
enhanced_dictionary.filter_extremes(
    no_below=5,
    no_above=0.8,
    keep_n=10000
)

# build raw bow corpus
enhanced_corpus = [enhanced_dictionary.doc2bow(doc) for doc in enhanced_lda_docs]

print(f"Dictionary size: {len(enhanced_dictionary)}")
print(f"Corpus size: {len(enhanced_corpus)}")

# apply TF-IDF model
tfidf = TfidfModel(enhanced_corpus)
enhanced_corpus_tfidf = tfidf[enhanced_corpus]

In [ ]:
# tuning number of topics and picking the one with the highest coherence

coherence_scores = []

# specify range of topics
min_topics = 5
max_topics = 60
step_size = 5
topics_range = range(min_topics, max_topics + 1, step_size)

for num_topics in topics_range:
    lda_model = LdaModel(
        corpus=enhanced_corpus_tfidf,
        id2word=enhanced_dictionary,
        num_topics=num_topics,
        passes=10,
        alpha="auto", # letting the model choose the alpha, eta
        eta="auto",
        random_state=42
    )
    cm = CoherenceModel(
        model=lda_model,
        texts=enhanced_lda_docs,
        dictionary=enhanced_dictionary,
        coherence="c_v"
    )
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"Topics: {num_topics}, Coherence: {score:.4f}")

# Plot
optimal_num_topics_enhanced = list(topics_range)[np.argmax(coherence_scores)]

plt.figure(figsize=(10, 6))
plt.plot(topics_range, coherence_scores, marker="o", color="b", label="Coherence Score")
plt.axvline(x=optimal_num_topics_enhanced, color="r", linestyle="--",
            label=f"Optimal: {optimal_num_topics_enhanced}")
plt.xlabel("Number of Topics")
plt.ylabel("Coherence Score")
plt.title("Enhanced LDA: Number of Topics vs Coherence Score")
plt.xticks(topics_range)
plt.legend()
plt.grid(True)
plt.show()

print(f"Optimal Number of Topics: {optimal_num_topics_enhanced}")
print(f"Best Coherence Score: {max(coherence_scores):.4f}")

In [ ]:
optimal_num_topics_enhanced = 30

In [ ]:
# enhanced LDA model

enhanced_lda_model = LdaModel(
    corpus=enhanced_corpus_tfidf,
    id2word=enhanced_dictionary,
    num_topics=optimal_num_topics_enhanced,
    passes=15,          # more passes for final model
    alpha="auto",
    eta="auto",
    random_state=42
)

# Preview topics
for idx, topic in enhanced_lda_model.print_topics(num_words=10):
    print(f"Topic {idx}: {topic}")

final_coherence = CoherenceModel(
    model=enhanced_lda_model,
    texts=enhanced_lda_docs,
    dictionary=enhanced_dictionary,
    coherence="c_v"
).get_coherence()

print(f"Final Enhanced LDA Coherence: {final_coherence:.4f}")

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning)

pyLDAvis.enable_notebook()

vis = gensimvis.prepare(
    enhanced_lda_model,
    enhanced_corpus_tfidf,
    enhanced_dictionary,
    sort_topics=False
)

pyLDAvis.display(vis)

In [ ]:
def plot_top_words(model, num_topics, num_words=8, topics_per_row=5):
    num_rows = math.ceil(num_topics / topics_per_row)
    fig, axes = plt.subplots(num_rows, topics_per_row,
                              figsize=(20, num_rows * 3))
    axes = axes.flatten()

    for topic_id in range(num_topics):
        words, weights = zip(*model.show_topic(topic_id, topn=num_words))
        axes[topic_id].barh(words, weights, color="steelblue")
        axes[topic_id].set_title(f"Topic {topic_id}", fontsize=10)
        axes[topic_id].invert_yaxis()

    # hide empty subplots
    for i in range(num_topics, len(axes)):
        axes[i].set_visible(False)

    plt.suptitle("Top Words per Topic", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig("topic_words_bar.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_top_words(enhanced_lda_model, num_topics=30)

In [ ]:
# get dominant topic for each chunk
def get_dominant_topic(model, corpus):
    dominant_topics = []
    for doc_bow in corpus:
        topics = model.get_document_topics(doc_bow)
        dominant = max(topics, key=lambda x: x[1])
        dominant_topics.append(dominant[0])
    return dominant_topics

enhanced_lda_df["dominant_topic"] = get_dominant_topic(
    enhanced_lda_model, enhanced_corpus_tfidf
)

# Plot topic distribution per season
topic_season = enhanced_lda_df.groupby(["season", "dominant_topic"]).size().unstack(fill_value=0)

topic_season.plot(
    kind="bar",
    figsize=(14, 6),
    colormap="tab20",
    legend=False
)
plt.title("Topic Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Number of Chunks")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("topic_season_distribution.png", dpi=300)
plt.show()

# Bertopic

For Bertopic, the best choice for the document is the use of scenes. Since the dataset does not split the text on scene natively, we experiment with chunking each document with x lines.

## Baseline

### Preprocessing

In [ ]:
bertopic_df = df.copy()
bertopic_df.head()

# clean and preprocess text

FILLERS = {'um', 'uh', 'hmm', 'oh', 'ah', 'okay', 'yeah', 'hey', 'like'}

def clean_for_bertopic(text):
    text = text.lower()
    text = re.sub(r"[^\w\s']", '', text)
    tokens = [t for t in text.split() if t not in FILLERS]
    return ' '.join(tokens)

bertopic_df["clean_text"] = bertopic_df["text"].apply(clean_for_bertopic)
bertopic_df = bertopic_df[bertopic_df["clean_text"].str.split().str.len() >= 3].reset_index(drop=True)
bertopic_df.head()

bertopic_df = create_chunked_corpus(bertopic_df, chunk_size=10, text_col='clean_text')
docs = bertopic_df['text'].tolist()
print(f"Total chunks: {len(docs)}")
bertopic_df.head()

### Model

In [ ]:


topic_model = BERTopic(
    language="english",
    nr_topics="auto",
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

### Evaluation

In [ ]:
# How many topics were found
topic_model.get_topic_info()

In [ ]:
# Top words for a specific topic
print(topic_model.get_topic(0))

# How many docs landed in the outlier topic (-1)
outlier_count = sum(t == -1 for t in topics)
total = len(topics)
print(f"Outliers: {outlier_count}/{total} ({outlier_count/total:.1%})")

In [ ]:
# top words for a specific topic
topic_model.get_topic(0)

In [ ]:
# add coherence score?

## Enhanced Bertopic

1. curated stopword list
2. restrict to nouns and adjectives
3. better embedding model
4. tune hdbscan
5. add mmr for topic diversity
6. find optimal number of topics
7. fit enhanced model



In [ ]:
enhanced_bertopic_df = bertopic_df.copy()
enhanced_bertopic_df

In [ ]:
# only keep nouns & adjectives (parallel to LDA)
nlp.max_length = 2000000

def pos_filter(text):
    doc = nlp(text, disable=['parser', 'ner'])
    tokens = [token.lemma_ for token in doc if token.pos_ in {'NOUN', 'PROPN', 'ADJ'}]
    return ' '.join(tokens)

stop_words_list = list(stop_words)
vectorizer = CountVectorizer(
    stop_words=stop_words_list,
    tokenizer=lambda text: pos_filter(text).split(),
    min_df=2
)

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    prediction_data=True
)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    random_state=42  # for reproducibility
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model_enhanced = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf_model,
    nr_topics='auto',
    verbose=True
)

In [ ]:
# fitting the model
topics, probs = topic_model_enhanced.fit_transform(docs)

outlier_count = sum(t == -1 for t in topics)
print(f"Outliers: {outlier_count}/{len(topics)} ({outlier_count/len(topics):.1%})")
print(topic_model_enhanced.get_topic_info())

In [ ]:
topic_model_enhanced.get_topic_info()

In [ ]:
results = []
combos = [(mcs, ms) for mcs in [5, 10, 15, 20] for ms in [1, 3, 5]]
total = len(combos)

for i, (min_cluster_size, min_samples) in enumerate(combos, 1):
    print(f"[{i}/{total}] Testing min_cluster_size={min_cluster_size}, min_samples={min_samples}...", end=' ')

    hdbscan_test = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, prediction_data=True)
    test_model = BERTopic(hdbscan_model=hdbscan_test, verbose=False)
    test_topics, _ = test_model.fit_transform(docs)

    outlier_rate = sum(t == -1 for t in test_topics) / len(test_topics)
    n_topics = len(set(test_topics)) - 1
    results.append({
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'outlier_rate': round(outlier_rate, 3),
        'n_topics': n_topics
    })
    print(f"done — outlier rate: {outlier_rate:.1%}, topics found: {n_topics}")

print("\nGrid search complete!")
results_df = pd.DataFrame(results).sort_values('outlier_rate')
results_df

In [ ]:
# delete once you run LDA
optimal_num_topics_enhanced = 30

In [ ]:
min_cluster_size = 15
min_samples = 1

# HDBSCAN with optimal parameters from grid search
hdbscan_model = HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    prediction_data=True
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)


# build the enhanced model with nr_topics matching LDA
topic_model_enhanced = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf_model,
    nr_topics=optimal_num_topics_enhanced,
    verbose=True
)

topics, probs = topic_model_enhanced.fit_transform(docs)

outlier_count = sum(t == -1 for t in topics)
print(f"Outliers: {outlier_count}/{len(topics)} ({outlier_count/len(topics):.1%})")
topic_model_enhanced.get_topic_info()

In [ ]:
# calculating coherence

tokenized_docs = [doc.split() for doc in docs]
dictionary = corpora.Dictionary(tokenized_docs)

# extract topic words, excluding the outlier topic (-1)
topic_words = [
    [word for word, _ in topic_model_enhanced.get_topic(t)]
    for t in range(topic_model_enhanced.get_topic_info().shape[0] - 1)
]

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

score = coherence_model.get_coherence()
print(f"Coherence Score (c_v): {score:.4f}")

In [ ]:
# coherence model npmi
coherence_model_npmi = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_npmi'
)
print(f"BERTopic c_npmi: {coherence_model_npmi.get_coherence():.4f}")

In [ ]:
def topic_diversity(topic_words, topk=10):
    unique_words = set()
    total_words = 0
    for words in topic_words:
        unique_words.update(words[:topk])
        total_words += min(len(words), topk)
    return len(unique_words) / total_words

diversity = topic_diversity(topic_words)
print(f"Topic Diversity: {diversity:.4f}")

In [ ]:
topic_info = topic_model_enhanced.get_topic_info()
topic_info_no_outlier = topic_info[topic_info['Topic'] != -1]

topic_info_no_outlier.plot(
    x='Topic', y='Count', kind='bar', figsize=(15, 4),
    title='Document Count per Topic', legend=False
)
plt.xlabel('Topic')
plt.ylabel('Document Count')
plt.tight_layout()
plt.show()

In [ ]:
for topic_id in range(min(30, len(topic_words))):
    words = [w for w, _ in topic_model_enhanced.get_topic(topic_id)][:10]
    print(f"Topic {topic_id}: {', '.join(words)}")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_coherence(topic_words, model):
    scores = []
    for words in topic_words:
        if len(words) < 2:
            continue
        embeddings = model.encode(words)
        # average pairwise cosine similarity
        sim_matrix = np.inner(embeddings, embeddings)
        n = len(words)
        score = (sim_matrix.sum() - n) / (n * (n - 1))  # exclude diagonal
        scores.append(score)
    return np.mean(scores)

emb_coherence = embedding_coherence(topic_words, embedding_model)
print(f"Embedding Coherence: {emb_coherence:.4f}")